# 5. Application Streamlit — Détection EPI

Ce notebook crée et lance l'application Streamlit de démonstration (Axe E).  
Exécuter les cellules dans l'ordre : les deux premières écrivent les fichiers `.py`, la troisième lance le serveur.

L'application sera accessible sur **http://51.15.234.209:8501**

In [ ]:
%%writefile /root/Projet_Image/epi_pipeline.py
import cv2
import numpy as np

TOUTES_REGLES = {
    'helmet':      {'partie_corps': 'head',   'message': 'casque manquant'},
    'safety-vest': {'partie_corps': 'person', 'message': 'gilet de securite manquant'},
    'gloves':      {'partie_corps': 'hands',  'message': 'gants manquants'},
}


def iou(boite_a, boite_b):
    xa1, ya1, xa2, ya2 = boite_a
    xb1, yb1, xb2, yb2 = boite_b
    x1, y1 = max(xa1, xb1), max(ya1, yb1)
    x2, y2 = min(xa2, xb2), min(ya2, yb2)
    inter  = max(0, x2 - x1) * max(0, y2 - y1)
    aire_a = (xa2 - xa1) * (ya2 - ya1)
    aire_b = (xb2 - xb1) * (yb2 - yb1)
    union  = aire_a + aire_b - inter
    return inter / union if union > 0 else 0.0


def verifier_conformite(detections, seuil_iou=0.1, regles=None):
    if regles is None:
        regles = TOUTES_REGLES
    motifs = []
    for classe_epi, regle in regles.items():
        boites_partie = [d['boite'] for d in detections if d['classe'] == regle['partie_corps']]
        boites_epi    = [d['boite'] for d in detections if d['classe'] == classe_epi]
        for boite_partie in boites_partie:
            protege = any(iou(boite_partie, boite_epi) > seuil_iou for boite_epi in boites_epi)
            if not protege:
                motifs.append(regle['message'])
    return len(motifs) > 0, list(dict.fromkeys(motifs))


def incruster_resultats(image, detections, non_conforme, motifs, conf_min=0.4):
    img = image.copy()
    h, w = img.shape[:2]
    for det in detections:
        if det['confiance'] < conf_min:
            continue
        x1, y1, x2, y2 = map(int, det['boite'])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 2)
        cv2.putText(img, f"{det['classe']} {det['confiance']:.2f}", (x1, max(y1 - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
    if non_conforme:
        cv2.rectangle(img, (0, 0), (w, 36), (0, 0, 255), -1)
        texte = "NON CONFORME : " + ", ".join(motifs)
        cv2.putText(img, texte, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return img

In [ ]:
%%writefile /root/Projet_Image/app.py
import sys
sys.path.insert(0, '/root/Projet_Image')

import streamlit as st
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

from epi_pipeline import TOUTES_REGLES, verifier_conformite, incruster_resultats

WEIGHTS = Path('/root/Projet_Image/runs/yolov8m_epi_1024-2/weights/best.pt')


@st.cache_resource
def charger_modele():
    return YOLO(str(WEIGHTS))


st.set_page_config(page_title="Détection EPI", layout="centered")
st.title("Détection EPI — chantier")
st.markdown("Importez une image de chantier pour vérifier la conformité EPI.")

# --- Sidebar ---
st.sidebar.header("Paramètres")
conf_min  = st.sidebar.slider("Confiance minimale", 0.1, 0.9, 0.4, 0.05)
seuil_iou = st.sidebar.slider("Seuil IoU", 0.05, 0.5, 0.1, 0.05)

st.sidebar.subheader("EPI à vérifier")
check_casque = st.sidebar.checkbox("Casque", value=True)
check_gilet  = st.sidebar.checkbox("Gilet de sécurité", value=True)
check_gants  = st.sidebar.checkbox("Gants", value=False)

# --- Upload ---
uploaded = st.file_uploader("Image (jpg / jpeg / png)", type=["jpg", "jpeg", "png"])

if uploaded is not None:
    model = charger_modele()

    arr   = np.frombuffer(uploaded.read(), np.uint8)
    image = cv2.imdecode(arr, cv2.IMREAD_COLOR)

    regles_actives = {}
    if check_casque:
        regles_actives['helmet'] = TOUTES_REGLES['helmet']
    if check_gilet:
        regles_actives['safety-vest'] = TOUTES_REGLES['safety-vest']
    if check_gants:
        regles_actives['gloves'] = TOUTES_REGLES['gloves']

    resultat   = model.predict(image, conf=conf_min, verbose=False)[0]
    detections = [
        {'classe': model.names[int(c)], 'boite': b.tolist(), 'confiance': float(conf)}
        for b, c, conf in zip(
            resultat.boxes.xyxy.cpu().numpy(),
            resultat.boxes.cls.cpu().numpy(),
            resultat.boxes.conf.cpu().numpy(),
        )
    ]

    non_conforme, motifs = verifier_conformite(detections, seuil_iou, regles_actives)
    image_annotee = incruster_resultats(image, detections, non_conforme, motifs, conf_min)

    if non_conforme:
        st.error("NON CONFORME : " + ", ".join(motifs))
    else:
        st.success("CONFORME")

    st.image(cv2.cvtColor(image_annotee, cv2.COLOR_BGR2RGB), use_container_width=True)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'streamlit', '-q'], check=True)
print('streamlit installé')

In [ ]:
import subprocess
proc = subprocess.Popen(
    ['streamlit', 'run', '/root/Projet_Image/app.py',
     '--server.port', '8501', '--server.address', '0.0.0.0',
     '--server.headless', 'true'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('Streamlit lancé — http://51.15.234.209:8501')
print('PID :', proc.pid)